In [70]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv()

genai_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
LLM_MODELS = [
    "gemini-2.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemma-4-26b-a4b-it",
]

In [71]:
import time

def generate_llm_response(prompt, system_instruction=None):
    last_error = None
    config = None
    if system_instruction:
        config = types.GenerateContentConfig(system_instruction=system_instruction)

    for model in LLM_MODELS:
        for attempt in range(3):
            try:
                return genai_client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=config,
                )
            except errors.ClientError as e:
                last_error = e
                if e.code in (429, 503):
                    time.sleep(2 ** attempt)
                    continue
                raise
            except errors.ServerError as e:
                last_error = e
                time.sleep(2 ** attempt)
                continue

    raise last_error

def llm(prompt, system_instruction=None):
    return generate_llm_response(prompt, system_instruction=system_instruction).text

In [72]:
llm("Hey, what's up?")

"Not much! Just hanging out in the digital void, ready to help you with whatever you need. How’s your day going? What's on your mind?"

In [73]:
# Simple LLM call without RAG context
answer = llm("Hey, what's up?")
print(answer)

Hey! Not too much, just here and ready to help. How about you? What's on your mind? Anything I can assist you with today?


In [74]:
question = "I just discovered the course. Can I still join?"

context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [75]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)

Yes, you can still join, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [76]:
answer = llm(prompt)
print(answer)

Yes, you can still join, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [77]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
docs_response = requests.get(docs_url)
courses_raw = docs_response.json()

In [78]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1346

In [79]:
documents[1000]

{'id': '356445714b',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 3. Machine Learning for Classification',
 'question': 'DictVectorizer: Fitting on validation data',
 'answer': "Validation datasets are used to optimize models by providing an estimate of performance on unseen data. Understanding how to properly use the `DictVectorizer` class is crucial for maintaining this separation between training and validation.\n\n- **Fitting on Training Data**: The `fit` method of `DictVectorizer` analyzes the training dataset to determine how to map dictionary values. Categorical features are one-hot encoded, while numeric features remain unchanged.\n- **Avoid Fitting on Validation Data**: Applying the `fit` method to validation data can lead to information leakage, as it exposes the model to data it should not see during training.\n- **Appropriate Usage**:\n  1. Use `fit_transform` on the training dataset.\n  2. Use `transform` only on validation and test datasets.\n\nBy following

In [80]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [81]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [82]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [83]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [84]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "llm-zoomcamp"}
)

In [85]:
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?']

In [86]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [87]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [88]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [89]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [90]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [91]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [92]:
llm_response = generate_llm_response(prompt)
answer = llm_response.text
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.


In [93]:
if "llm_response" not in globals():
    llm_response = generate_llm_response(prompt)

answer = llm_response.text
answer

'Yes, you can still join the course. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.'

In [94]:
if "llm_response" not in globals():
    llm_response = generate_llm_response(prompt)

llm_response.candidates[0]

Candidate(
  content=Content(
    parts=[
      Part(
        text='Yes, you can still join the course. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.',
        thought_signature=b'\x124\n2\x01\x0c9\xd6\xc7\xe0\x80\xe9\xb2\xd3aq\xa4\x1d\x1b\x06\x9e\x81R!G6@\xb6E\xcb\x98\x1a\x16\x9c*\x18\x15\x8c\x81{\xbe\x12\x80\x97\x16Rr\xafn\x8e]\x1a\xc9j'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)

In [95]:
if "llm_response" not in globals():
    llm_response = generate_llm_response(prompt)

llm_response.candidates[0].content.parts[0].text

'Yes, you can still join the course. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.'

In [97]:
if "llm_response" not in globals():
    llm_response = generate_llm_response(prompt)

llm_response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=31,
  prompt_token_count=514,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=514
    ),
  ],
  total_token_count=545
)

In [99]:
if "llm_response" not in globals():
    llm_response = generate_llm_response(prompt)

usage = llm_response.usage_metadata
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    usage.prompt_token_count * input_price +
    usage.candidates_token_count * output_price
)

cost

0.000525

In [104]:
import os
import time
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

if "genai_client" not in globals():
    load_dotenv()
    genai_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

if "LLM_MODELS" not in globals():
    LLM_MODELS = [
        "gemini-2.5-flash-lite",
        "gemini-3.1-flash-lite",
        "gemma-4-26b-a4b-it",
    ]

message_history = [
    {"role": "developer", "content": INSTRUCTIONS.strip()},
    {"role": "user", "content": prompt},
]

config = types.GenerateContentConfig(
    system_instruction=message_history[0]["content"],
)

llm_response = None
last_error = None
for model in LLM_MODELS:
    for attempt in range(3):
        try:
            llm_response = genai_client.models.generate_content(
                model=model,
                contents=message_history[1]["content"],
                config=config,
            )
            break
        except errors.ClientError as e:
            last_error = e
            if e.code in (429, 503):
                time.sleep(2 ** attempt)
                continue
            raise
        except errors.ServerError as e:
            last_error = e
            time.sleep(2 ** attempt)
            continue
    if llm_response is not None:
        break

if llm_response is None:
    raise last_error

answer = llm_response.text
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [106]:
def llm(instructions, user_prompt, model=None):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    config = types.GenerateContentConfig(
        system_instruction=message_history[0]["content"],
    )

    models = LLM_MODELS if model is None else [model] + [
        m for m in LLM_MODELS if m != model
    ]

    last_error = None
    for m in models:
        for attempt in range(3):
            try:
                response = genai_client.models.generate_content(
                    model=m,
                    contents=message_history[1]["content"],
                    config=config,
                )
                return response.text
            except errors.ClientError as e:
                last_error = e
                if e.code in (429, 503):
                    time.sleep(2 ** attempt)
                    continue
                raise
            except errors.ServerError as e:
                last_error = e
                time.sleep(2 ** attempt)
                continue

    raise last_error

In [107]:
def rag(query, model=None):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [108]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [109]:
rag("How do I get a certificate?")

'To get a certificate, you must follow these requirements:\n\n*   **Finish with a "live" cohort:** You cannot get a certificate for self-paced mode.\n*   **Peer-review:** You must peer-review 3 capstone projects after submitting your own. Peer-reviewing is only possible while the course is running.\n*   **Pass the Capstone project:** Passing the Capstone project is mandatory to receive the certificate (homework is not mandatory).\n*   **Update your profile:** Ensure your official name (as it appears on your identification documents) is entered in the second field under "Edit Course Profile," otherwise the default placeholder name will appear on your certificate.'